### Transaction location profiling

In [0]:
%sql
SELECT
    CASE
        WHEN is_online = true
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'A - ONLINE: state and zip missing'

        WHEN is_online = true
             AND (merchant_state IS NOT NULL OR merchant_zip IS NOT NULL)
            THEN 'A - ONLINE: UNEXPECTED LOCATION DATA'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND merchant_state RLIKE '^[A-Za-z]{2}$'
             AND merchant_zip IS NOT NULL
            THEN 'US - state + zip populated'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND NOT (merchant_state RLIKE '^[A-Za-z]{2}$')
             AND merchant_zip IS NULL
            THEN 'B - INTERNATIONAL: country + zip missing'

        WHEN is_online = false
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'UNEXPECTED: offline with no location'

        ELSE 'OTHER / POTENTIAL DATA QUALITY ISSUE'
    END AS location_pattern,
    COUNT(*) AS transaction_count
FROM credit_card_project.silver.cleaned_transactions
GROUP BY
    CASE
        WHEN is_online = true
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'A - ONLINE: state and zip missing'

        WHEN is_online = true
             AND (merchant_state IS NOT NULL OR merchant_zip IS NOT NULL)
            THEN 'A - ONLINE: UNEXPECTED LOCATION DATA'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND merchant_state RLIKE '^[A-Za-z]{2}$'
             AND merchant_zip IS NOT NULL
            THEN 'US - state + zip populated'

        WHEN is_online = false
             AND merchant_state IS NOT NULL
             AND NOT (merchant_state RLIKE '^[A-Za-z]{2}$')
             AND merchant_zip IS NULL
            THEN 'B - INTERNATIONAL: country + zip missing'

        WHEN is_online = false
             AND merchant_state IS NULL
             AND merchant_zip IS NULL
            THEN 'UNEXPECTED: offline with no location'

        ELSE 'OTHER / POTENTIAL DATA QUALITY ISSUE'
    END
ORDER BY transaction_count DESC;


### Merchant identity and location
Does each merchant have exactly one MCC and one location, or can those attributes vary?

In [0]:
%sql

SELECT
    merchant_id,
    COUNT(*) AS transaction_count,
    COUNT(DISTINCT merchant_category_code) AS distinct_mccs,
    COUNT(DISTINCT merchant_city) AS distinct_cities,
    COUNT(DISTINCT merchant_state) AS distinct_states,
    COUNT(DISTINCT merchant_zip) AS distinct_zips
FROM credit_card_project.silver.cleaned_transactions
GROUP BY merchant_id
HAVING
    COUNT(DISTINCT merchant_category_code) > 1
    OR COUNT(DISTINCT merchant_city) > 1
    OR COUNT(DISTINCT merchant_state) > 1
    OR COUNT(DISTINCT merchant_zip) > 1
ORDER BY transaction_count DESC;


Result: For these high-volume merchants, distinct_mccs = 1 consistently, while cities, states, and ZIPs can vary dramatically.

Next: Is MCC stable for a merchant across the entire dataset?

In [0]:
%sql
SELECT
    COUNT(*) AS total_merchants,
    SUM(CASE WHEN distinct_mccs = 1 THEN 1 ELSE 0 END) AS merchants_single_mcc,
    SUM(CASE WHEN distinct_mccs > 1 THEN 1 ELSE 0 END) AS merchants_multiple_mccs,
    MAX(distinct_mccs) AS max_mccs_per_merchant
FROM (
    SELECT
        merchant_id,
        COUNT(DISTINCT merchant_category_code) AS distinct_mccs
    FROM credit_card_project.silver.cleaned_transactions
    GROUP BY merchant_id
) m;


Result: So merchant_id → MCC is almost, but not completely, one-to-one.

Are above 654 exceptions (merchants_multiple_mccs) genuinely associated with two MCCs or is there's some temporal/data pattern behind it?

In [0]:
%sql
SELECT
    merchant_id,
    merchant_category_code,
    COUNT(*) AS transaction_count,
    MIN(transaction_timestamp) AS first_transaction,
    MAX(transaction_timestamp) AS last_transaction
FROM credit_card_project.silver.cleaned_transactions
WHERE merchant_id IN (
    SELECT merchant_id
    FROM (
        SELECT
            merchant_id,
            COUNT(DISTINCT merchant_category_code) AS distinct_mccs
        FROM credit_card_project.silver.cleaned_transactions
        GROUP BY merchant_id
    )
    WHERE distinct_mccs > 1
)
GROUP BY
    merchant_id,
    merchant_category_code
ORDER BY
    merchant_id,
    transaction_count DESC;


Finding: merchant → MCC is not strictly one-to-one, so merchant_category_code should remain a transaction-level analytical attribute and we should not put MCC into a merchant dimension as though it were a fixed merchant attribute.

=> merchant_id alone and its metrics should aggregate transactions regardless of MCC

Does every transaction have a merchant ID?

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(merchant_id) AS transactions_with_merchant_id,
    COUNT(*) - COUNT(merchant_id) AS transactions_without_merchant_id,
    COUNT(DISTINCT merchant_id) AS distinct_merchants
FROM credit_card_project.silver.cleaned_transactions;


Merchant profiling conclusion:
- 24,386,900 / 24,386,900 transactions have a merchant_id.
- There are 100,343 distinct merchants.
- Merchant IDs can be negative or positive; sign has no business meaning.
- A merchant can transact across many cities, states/countries, and ZIP codes.
- merchant_id is therefore the appropriate merchant identifier.
- merchant_category_code is not strictly a fixed merchant attribute because 654 merchants have multiple MCCs.
- MCC should therefore remain a transaction-level/category attribute, rather than being treated as a permanent property of the merchant.![](path)

### Next profiling target: users
profile of the user → card → transaction relationships.

Does every user have at least one card, and does every card have transactions?

In [0]:
%sql
SELECT
    COUNT(*) AS total_users,
    COUNT(CASE WHEN card_count > 0 THEN 1 END) AS users_with_cards,
    COUNT(CASE WHEN card_count = 0 THEN 1 END) AS users_without_cards,
    MIN(card_count) AS min_cards_per_user,
    MAX(card_count) AS max_cards_per_user,
    AVG(card_count) AS avg_cards_per_user
FROM (
    SELECT
        u.user_id,
        COUNT(c.card_id) AS card_count
    FROM credit_card_project.bronze.users u
    LEFT JOIN credit_card_project.bronze.cards c
        ON u.user_id = c.user_id
    GROUP BY u.user_id
) x;


User → Card profiling conclusion

We can now establish:

- 2,000 / 2,000 users have at least one card
- No users without cards
- Minimum cards per user: 1
- Maximum: 9
- Average: 3.073 cards per user

In [0]:
%sql
SELECT
    COUNT(*) AS total_cards,
    COUNT(CASE WHEN transaction_count > 0 THEN 1 END) AS cards_with_transactions,
    COUNT(CASE WHEN transaction_count = 0 THEN 1 END) AS cards_without_transactions,
    MIN(transaction_count) AS min_transactions_per_card,
    MAX(transaction_count) AS max_transactions_per_card,
    AVG(transaction_count) AS avg_transactions_per_card
FROM (
    SELECT
        c.user_id,
        c.card_id,
        COUNT(t.card_id) AS transaction_count
    FROM credit_card_project.bronze.cards c
    LEFT JOIN credit_card_project.silver.cleaned_transactions t
        ON c.user_id = t.user_id
        AND c.card_id = t.card_id
    GROUP BY
        c.user_id,
        c.card_id
) x;


Card profiling conclusion

We have:

- 6,146 total cards
- 6,139 cards with transactions
- 7 cards with zero transactions
- Minimum transactions per card: 0
- Maximum: 70,008
- Average: 3,967.93

So there are exactly 7 cards that exist in the source but have never appeared in the transaction data.
=> That means gold_card_behavior should be card-driven, not transaction-driven.